In [ ]:
pip install sentence-transformers pandas


In [ ]:
# 1. 패키지 불러오기 (설치가 안 되었다면 !pip install sentence-transformers 실행)
from sentence_transformers import SentenceTransformer
import pandas as pd

# 2. 모델 로드 (다국어 지원 모델 - 한글 검색을 위해 필수!)
# [왜 이 모델인가요?]: 한국어와 영어를 동시에 이해하는 똑똑한 AI 모델입니다.
print("🧠 AI 모델 로딩 중... 잠시만 기다려주세요.")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("✅ 모델 로드 완료!")

# 3. 테스트 데이터
test_games = [
    "위쳐 3: 괴물 사냥꾼이 되어 모험을 떠나는 다크 판타지 RPG",
    "스타듀 밸리: 시골 마을에서 농사를 짓고 힐링하는 게임",
    "사이버펑크 2077: 미래 도시에서 펼쳐지는 SF 액션"
]

# 4. 임베딩(벡터 변환) 실행
print("🔄 텍스트를 AI 숫자(벡터)로 변환 중...")
embeddings = model.encode(test_games)

# 5. 결과 확인
print(f"▶ 변환된 문장 개수: {len(embeddings)}개")
print(f"▶ 첫 번째 벡터 샘플 (5개 숫자만): {embeddings[0][:5]}")

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 유저의 검색어 임베딩 (영어가 아니어도 됨!)
user_query = "괴물을 사냥하는 어두운 분위기 모험"
query_vector = model.encode([user_query])

# 2. 게임 데이터들과의 유사도 계산
# [왜 이 기술을 쓰나요?]: 유저 검색어(숫자)와 게임 리스트(숫자)가 
# 얼마나 '의미적으로 닮았는지' 0~1 사이의 점수로 뽑아내기 위해서입니다.
distances = cosine_similarity(query_vector, embeddings)

# 3. 결과 보기 좋게 출력
print(f"🔍 유저 검색어: '{user_query}'")
print("-" * 30)

for i, game in enumerate(test_games):
    score = distances[0][i]
    print(f"🎮 게임: {game}")
    print(f"📊 유사도 점수: {score:.4f}")
    print("-" * 30)

🔍 유저 검색어: '괴물을 사냥하는 어두운 분위기 모험'
------------------------------
🎮 게임: 위쳐 3: 괴물 사냥꾼이 되어 모험을 떠나는 다크 판타지 RPG
📊 유사도 점수: 0.6682
------------------------------
🎮 게임: 스타듀 밸리: 시골 마을에서 농사를 짓고 힐링하는 게임
📊 유사도 점수: 0.4276
------------------------------
🎮 게임: 사이버펑크 2077: 미래 도시에서 펼쳐지는 SF 액션
📊 유사도 점수: 0.4113
------------------------------


In [5]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 오늘 작업할 파트: 데이터의 '질' 높이기 (Data Enrichment) 
    openai에 ai 라벨링을 통해 잘 되는지 확인

In [11]:
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv

# 1. 업스테이지 연결 설정 (OpenAI SDK와 호환됨)
# [왜 이렇게 바꾸나요?]: 업스테이지의 Solar 모델을 사용하기 위해 base_url을 변경합니다.
# Solar는 한국어 게임 설명을 훨씬 더 정확하게 분석해서 '매니아 점수'를 뽑아줍니다.
load_dotenv() 
MY_KEY = os.getenv("UPSTAGE_API_KEY")

client = OpenAI(
    api_key=MY_KEY, # 💡 정답! (따옴표를 지우고, 위에서 만든 변수 MY_KEY를 넣어야 함)
    base_url="https://api.upstage.ai/v1/solar"
)

# 2. 데이터 로드 (경로가 다르면 수정)
file_path = r"C:\div_log_kjt\hidden-gem-devlog\data\hidden_gem_bulk_data.csv"
df = pd.read_csv(file_path)

# 3. AI 점수 측정 함수 (강화 버전)
def ask_upstage_score(game_name):
    # [왜 이 기술을 쓰나요?]: 단순히 점수만 묻는 게 아니라, 
    # '시스템 프롬프트'를 통해 AI에게 명확한 전문가 페르소나를 부여합니다.
    try:
        response = client.chat.completions.create(
            model="solar-mini", # 업스테이지의 가성비 모델
            messages=[
                {
                    "role": "system", 
                    "content": "너는 게임 전문 평론가야. 유저가 게임 이름을 주면, 해당 게임이 '대중성은 낮으나 매니아층이 열광하는 숨겨진 명작'인지 판단해서 1~10점 사이의 숫자만 답해. 숫자가 높을수록 숨겨진 명작이야."
                },
                {"role": "user", "content": f"게임 이름: {game_name}"}
            ],
            max_tokens=5, # 숫자만 받으면 되니까 토큰을 최소화해서 비용 절약
            temperature=0  # 일관된 점수를 위해 무작위성을 0으로 설정
        )
        # 결과값에서 숫자만 추출
        score_str = response.choices[0].message.content.strip()
        return int(score_str)
    except Exception as e:
        # [에러 방어]: 수십만 개 돌릴 때 하나 에러 났다고 멈추면 안 되니까 기본 점수(5)를 줍니다.
        print(f"⚠️ {game_name} 처리 중 오류 발생: {e}")
        return 5

# 4. 딱 5개만 먼저 테스트 (실전!)
print("🚀 Upstage AI 라벨링 테스트 시작...")
df_sample = df.head(5).copy()
df_sample['mania_score'] = df_sample['name'].apply(ask_upstage_score)

print("\n✅ 결과 확인:")
print(df_sample[['name', 'mania_score']])

🚀 Upstage AI 라벨링 테스트 시작...
⚠️ The Elder Scrolls VI 처리 중 오류 발생: invalid literal for int() with base 10: '8\n\n"The'
⚠️ No Case Should Remain Unsolved 처리 중 오류 발생: invalid literal for int() with base 10: '이 게임 "No Case'
⚠️ Gimmick! 처리 중 오류 발생: invalid literal for int() with base 10: '이 게임 "Gimm'
⚠️ Super Robot Taisen: Original Generation 처리 중 오류 발생: invalid literal for int() with base 10: '8\n\nSuper Rob'
⚠️ The Witcher 3: Wild Hunt – Blood and Wine 처리 중 오류 발생: invalid literal for int() with base 10: '9\n\n"The'

✅ 결과 확인:
                                        name  mania_score
0                       The Elder Scrolls VI            5
1             No Case Should Remain Unsolved            5
2                                   Gimmick!            5
3    Super Robot Taisen: Original Generation            5
4  The Witcher 3: Wild Hunt – Blood and Wine            5


## open ai 를 통해 약 500 개의 게임의 점수를 매기려고 함

In [12]:
import pandas as pd
from openai import OpenAI
import os
import re # 💡 텍스트에서 숫자만 추출하기 위한 라이브러리 추가
from dotenv import load_dotenv

# 1. 연결 설정
load_dotenv() 
MY_KEY = os.getenv("UPSTAGE_API_KEY")

client = OpenAI(
    api_key=MY_KEY, 
    base_url="https://api.upstage.ai/v1/solar"
)

# 2. 데이터 로드 
file_path = r"C:\div_log_kjt\hidden-gem-devlog\data\hidden_gem_bulk_data.csv"
df = pd.read_csv(file_path)

# 3. AI 점수 측정 함수 (숫자 추출기 장착!)
def ask_upstage_score(game_name):
    try:
        response = client.chat.completions.create(
            model="solar-mini", 
            messages=[
                {
                    "role": "system", 
                    "content": "너는 게임 전문 평론가야. 유저가 게임 이름을 주면, 해당 게임이 '대중성은 낮으나 매니아층이 열광하는 숨겨진 명작'인지 판단해. 설명은 절대 하지 말고 오직 1에서 10 사이의 숫자 하나만 출력해." # 프롬프트도 더 강력하게!
                },
                {"role": "user", "content": f"게임 이름: {game_name}"}
            ],
            max_tokens=10, # AI가 말을 시작할 수 있게 여유를 조금 줌
            temperature=0  
        )
        
        # AI의 대답 전체 가져오기
        ai_reply = response.choices[0].message.content.strip()
        
        # 💡 [핵심 기술]: 정규표현식으로 문장 안에서 연속된 숫자만 싹 다 찾기
        numbers = re.findall(r'\d+', ai_reply)
        
        if numbers:
            # 첫 번째로 찾은 숫자를 정수로 변환 (예: "8점입니다" -> 8)
            score = int(numbers[0])
            # 점수가 1~10을 벗어나지 않게 안전장치
            return max(1, min(10, score))
        else:
            return 5 # 숫자를 아예 안 썼을 경우 5점

    except Exception as e:
        print(f"⚠️ {game_name} 처리 중 오류 발생: {e}")
        return 5

# 4. 딱 5개만 먼저 테스트
print("🚀 Upstage AI 라벨링 테스트 시작 (숫자 추출기 가동)...")
df_sample = df.head(5).copy()
df_sample['mania_score'] = df_sample['name'].apply(ask_upstage_score)

print("\n✅ 결과 확인:")
print(df_sample[['name', 'mania_score']])

🚀 Upstage AI 라벨링 테스트 시작 (숫자 추출기 가동)...

✅ 결과 확인:
                                        name  mania_score
0                       The Elder Scrolls VI            8
1             No Case Should Remain Unsolved            8
2                                   Gimmick!            8
3    Super Robot Taisen: Original Generation            8
4  The Witcher 3: Wild Hunt – Blood and Wine            9


## 1000개정도 다양한 게임 장르의 데이터 벌크 만들자!

In [14]:
import requests
import pandas as pd
import time
import os
from tqdm import tqdm
from dotenv import load_dotenv

# [1. 환경 변수 세팅]
load_dotenv()
RAWG_API_KEY = os.getenv("RAWG_API_KEY")

if not RAWG_API_KEY:
    raise ValueError("🚨 .env 파일에 RAWG_API_KEY가 없습니다!")

# 💡 [핵심 수정]: 절대 경로 대신 현재 폴더 기준의 'data' 폴더로 변경
os.makedirs('data', exist_ok=True) # data 폴더가 없으면 자동으로 만들어줌!
SAVE_PATH = "data/hidden_gem_balanced_1000.csv"

# [2. 10대 주요 장르 설정 (다양성 확보)]
genres = ['action', 'indie', 'adventure', 'role-playing-games-rpg', 'strategy', 'shooter', 'casual', 'simulation', 'horror', 'puzzle']
all_games = []

print("🚀 [Ultimate Mode] 고품질 데이터 수집 파이프라인 가동 (목표: 순수 데이터 1,000개 이상)")

# [3. 초기 데이터 넉넉하게 수집 (장르당 5페이지 탐색)]
for genre in genres:
    for page in range(1, 6): 
        url = f"https://api.rawg.io/api/games?key={RAWG_API_KEY}&genres={genre}&page={page}&ordering=-added"
        try:
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                data = res.json()
                for game in data.get('results', []):
                    all_games.append({
                        'id': game['id'],
                        'name': game['name'],
                        'slug': game.get('slug'),
                        'rating': game.get('rating'),
                        'added': game.get('added'),
                        'genres': ", ".join([g['name'] for g in game['genres']])
                    })
        except Exception as e:
            print(f"⚠️ {genre} 장르 {page}페이지 수집 스킵: {e}")
        time.sleep(0.15) 

df_raw = pd.DataFrame(all_games).drop_duplicates(subset='id')
print(f"✅ 1차 수집 완료: 총 {len(df_raw)}개 후보군 확보")

# [4. 상세 설명(Description) 추출 및 퀄리티 컨트롤]
descriptions = []
print(f"\n🔍 AI 감정을 위한 '상세 설명' 딥다이브 추출 중... (대상: {len(df_raw)}개)")

for game_id in tqdm(df_raw['id']):
    try:
        detail_url = f"https://api.rawg.io/api/games/{game_id}?key={RAWG_API_KEY}"
        detail_data = requests.get(detail_url, timeout=10).json()
        desc = detail_data.get('description_raw') or ""
        descriptions.append(desc)
    except:
        descriptions.append("")
    time.sleep(0.15)

df_raw['description'] = descriptions

# [5. 깐깐한 필터링 및 최종 저장]
df_raw['description'] = df_raw['description'].fillna("")
df_final = df_raw[df_raw['description'].str.strip() != ""].copy()

df_final.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f"\n💎 [성공] 퀄리티 컨트롤 완료! 불량품 제외 총 **{len(df_final)}**개의 최상급 데이터 확보.")
print(f"📂 저장 위치: {SAVE_PATH}")

🚀 [Ultimate Mode] 고품질 데이터 수집 파이프라인 가동 (목표: 순수 데이터 1,000개 이상)
✅ 1차 수집 완료: 총 583개 후보군 확보

🔍 AI 감정을 위한 '상세 설명' 딥다이브 추출 중... (대상: 583개)


100%|██████████| 583/583 [11:06<00:00,  1.14s/it]


💎 [성공] 퀄리티 컨트롤 완료! 불량품 제외 총 **583**개의 최상급 데이터 확보.
📂 저장 위치: data/hidden_gem_balanced_1000.csv


In [16]:
# [제목: 절대 경로를 적용한 1,000개 데이터 퀄리티 자동 검증 스크립트]
# 설명: 파일 위치 혼동을 막기 위해 안전한 절대 경로를 사용하여 수집된 데이터의 상태를 검증합니다.

import pandas as pd
import os

# 💡 [핵심 수정]: 아까 네가 알려준 정확한 절대 경로로 고정!
file_path = r"C:\div_log_kjt\hidden-gem-devlog\data\hidden_gem_balanced_1000.csv"

# 파일이 진짜 그 위치에 있는지 먼저 검사하는 방어 로직
if not os.path.exists(file_path):
    print(f"🚨 앗! 지정한 경로에 파일이 없습니다. 경로를 다시 확인해주세요: {file_path}")
else:
    # 1. 수집된 데이터 불러오기
    df = pd.read_csv(file_path)

    print("📊 [데이터 검증 리포트]")
    print("-" * 40)

    # 2. 총 데이터 수 및 결측치 확인
    print(f"✅ 총 살아남은 게임 수: {len(df)}개")
    print(f"✅ 빈 설명(Null) 개수: {df['description'].isnull().sum()}개 (0이어야 정상!)")

    # 3. 텍스트 길이 검사
    df['desc_length'] = df['description'].apply(lambda x: len(str(x)))
    print(f"✅ 평균 설명 길이: {df['desc_length'].mean():.0f}자")
    print(f"⚠️ 100자 이하의 너무 짧은 설명 개수: {len(df[df['desc_length'] < 100])}개")

    # 4. 장르 밸런스 확인 (상위 5개)
    print("\n🔥 [가장 많이 수집된 장르 조합 Top 5]")
    print(df['genres'].value_counts().head(5))

📊 [데이터 검증 리포트]
----------------------------------------
✅ 총 살아남은 게임 수: 583개
✅ 빈 설명(Null) 개수: 0개 (0이어야 정상!)
✅ 평균 설명 길이: 1223자
⚠️ 100자 이하의 너무 짧은 설명 개수: 0개

🔥 [가장 많이 수집된 장르 조합 Top 5]
genres
Shooter, Action         61
Action, RPG             36
Adventure, Action       33
Strategy                16
Strategy, Simulation    15
Name: count, dtype: int64


## [제목: 12종 AI 정밀 감정 및 자동 저장 파이프라인 (583개 원석 가공)]

In [ ]:
import pandas as pd
import json
import time
import os
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# [1. 환경 설정 및 API 연결]
load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

if not UPSTAGE_API_KEY:
    raise ValueError("🚨 .env 파일에 UPSTAGE_API_KEY가 없습니다!")

client = OpenAI(
    api_key=UPSTAGE_API_KEY, 
    base_url="https://api.upstage.ai/v1/solar"
)

# [2. 입출력 절대 경로 설정]
input_path = r"C:\div_log_kjt\hidden-gem-devlog\data\hidden_gem_balanced_1000.csv"
output_path = r"C:\div_log_kjt\hidden-gem-devlog\data\hidden_gem_analyzed_final.csv"

# 데이터 로드
df = pd.read_csv(input_path)

# [3. AI가 채워넣을 12가지 지표 컬럼 초기화]
score_columns = [
    'mania_score', 'story_depth', 'narrative_power', 'character_charm', 
    'play_time', 'kindness', 'art_visual', 'optimization', 
    'difficulty', 'innovation', 'sound_bgm', 'gem_potential'
]

# 아직 분석 안 된 컬럼은 -1로 셋팅 (이어하기 기능 지원)
for col in score_columns:
    if col not in df.columns:
        df[col] = -1 

# [4. AI 보석 감정 함수]
def analyze_game_pro(game_name, description):
    try:
        response = client.chat.completions.create(
            model="solar-mini",
            messages=[
                {
                    "role": "system",
                    "content": """너는 게임 전문 데이터 분석가야. 게임 이름과 설명을 보고 아래 12가지 항목을 1~10점 사이 정수로 평가해서 오직 JSON 형식으로만 답해.
                    항목: mania_score, story_depth, narrative_power, character_charm, play_time, kindness, art_visual, optimization, difficulty, innovation, sound_bgm, gem_potential"""
                },
                # 너무 긴 텍스트는 토큰 절약을 위해 1500자 선에서 커트
                {"role": "user", "content": f"게임 이름: {game_name}\n설명: {str(description)[:1500]}"} 
            ],
            response_format={ "type": "json_object" },
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # 가끔 AI가 JSON 형식을 삐끗할 때를 대비한 방어
        print(f"⚠️ {game_name} 분석 중 에러: {e}")
        return None

# [5. 본격적인 583개 데이터 감정 시작]
print(f"🔥 총 {len(df)}개 원석 AI 정밀 감정 가동 (20개 단위 자동 저장 적용)")

for i, row in tqdm(df.iterrows(), total=len(df)):
    # 아직 분석 안 된(-1) 데이터만 쏙쏙 골라서 처리
    if row['mania_score'] == -1:
        result = analyze_game_pro(row['name'], row['description'])
        
        if result:
            for col in score_columns:
                # 결과값이 누락되었을 경우를 대비해 기본값 5 부여
                df.at[i, col] = result.get(col, 5) 
                
        time.sleep(0.1) # 서버 보호 매너 타임
        
        # 💡 [핵심 방어 로직]: 20개마다 하드디스크에 자동 덮어쓰기 저장!
        if i > 0 and i % 20 == 0:
            df.to_csv(output_path, index=False, encoding='utf-8-sig')

# [6. 모든 작업 완료 후 최종 저장]
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 보석 세공 완벽 종료! 결과 파일이 생성되었습니다: {output_path}")